# Device Analysis Report (CSV Exports)

This notebook extends the DisMAL `devices` report using additional Discovery Analysis exports.
It enriches each device identity with consistency, reason_not_updated, current access,
credential label/username lookups, and DiscoveryRun outpost details.


## Requirements

We rely on `pandas`, `numpy`, `PyYAML`, and optionally `tideway` for vault credential lookups.
Uncomment the cell below to install them if needed.


In [ ]:
# %pip install -q pandas numpy pyyaml tideway

import datetime as dt
import math
from ast import literal_eval
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml

try:
    import tideway
except Exception:
    tideway = None


## Configuration

Adjust these values to control where CSV exports are loaded from and where results are written.


In [ ]:
# Root folder that contains sub-folders per appliance (e.g., raw_exports/dev)
RAW_EXPORT_ROOT = Path("../raw_exports")
# Optional filters (set to a list like ["prod"] to limit processing)
INCLUDE_INSTANCES = None  # or list of appliance names
EXCLUDE_INSTANCES = []    # names to skip even if present in config
# Optional credential UUID filter (last segment, with or without prefix)
DEVICES_WITH_CRED_UUID = None  # e.g., "7636fe3b4bd69466ab487f0000010700"
# Optional base directory override for outputs (per appliance sub-folder is created automatically)
OUTPUT_BASE_DIR = None  # e.g., Path("../../csv_outputs")
OUTPUT_FILENAME = "device_analysis.csv"
# CSV filenames used by this notebook
CSV_IDENTITIES = "devices_report_identities.csv"
CSV_LAST_DISCOVERY = "devices_report_last_discovery.csv"
CSV_DISCOVERY_KEY_MAP = "discovery_analysis_key_map.csv"
CSV_DISCOVERY_ACCESS_SUMMARY = "discovery_analysis_access_summary.csv"
CSV_DISCOVERY_DEVICEINFO = "discovery_analysis_deviceinfo.csv"
CSV_DISCOVERY_RUNS = "discovery_analysis_discovery_runs.csv"
CSV_DISCOVERY_DROPPED = "discovery_analysis_dropped_endpoints.csv"
CSV_OUTPOST_MAP_OPTIONAL = "discoveryrun_outpost_map.csv"  # optional
CSV_CREDENTIAL_SUCCESS_FALLBACK = "credential_success.csv"  # optional fallback from output_<target>


In [ ]:
def find_repo_root(start: Path) -> Path:
    """
    Return the nearest parent directory that looks like the repo root.
    
    This keeps the notebook portable so relative paths resolve correctly even when the working directory changes.
    """
    for candidate in [start] + list(start.parents):
        if (candidate / "config.yaml").exists() or (candidate / ".git").is_dir():
            return candidate
    return start


def parse_config_bool(value, default: bool = True) -> bool:
    """
    Normalize boolean-like config values from YAML/CLI into a strict bool.
    
    This avoids accidental truthiness bugs when values arrive as strings such as "true" or "false".
    """
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    return default


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
CONFIG_PATH = REPO_ROOT / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"config.yaml not found at {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as fh:
    cfg = yaml.safe_load(fh) or {}

appliance_entries = cfg.get("appliances") or []
if isinstance(appliance_entries, dict):
    appliance_entries = [appliance_entries]

if not appliance_entries:
    default_target = cfg.get("target")
    default_name = cfg.get("name") or (default_target or "default")
    appliance_entries = [{"name": default_name, "target": default_target}]


def resolve_token(entry: Dict[str, Any]) -> Optional[str]:
    """
    Resolve an appliance API token from inline config or token file fallback.
    
    Centralizing this logic ensures consistent auth behavior across all selected appliances.
    """
    token = str(entry.get("token") or cfg.get("token") or "").strip()
    if token:
        return token

    token_file = entry.get("token_file") or cfg.get("token_file") or cfg.get("f_token")
    if not token_file:
        return None

    token_path = Path(token_file)
    if not token_path.is_absolute():
        token_path = REPO_ROOT / token_path
    if not token_path.exists():
        return None

    text = token_path.read_text(encoding="utf-8").strip()
    return text or None


available_appliances: List[Dict[str, Any]] = []
for entry in appliance_entries:
    name = str(entry.get("name") or "").strip()
    target = str(entry.get("target") or "").strip()
    if not name:
        continue

    token = resolve_token(entry)
    api_version = str(entry.get("api_version") or cfg.get("api_version") or "v1.14")
    verify_ssl = parse_config_bool(entry.get("verify_ssl", cfg.get("verify_ssl", True)), default=True)

    available_appliances.append({
        "name": name,
        "target": target,
        "token": token,
        "api_version": api_version,
        "verify_ssl": verify_ssl,
    })

if not available_appliances:
    raise ValueError("No appliances with a name found in config.yaml")

exports_root = RAW_EXPORT_ROOT if RAW_EXPORT_ROOT.is_absolute() else (NOTEBOOK_DIR / RAW_EXPORT_ROOT).resolve()
if not exports_root.exists():
    raise FileNotFoundError(f"Raw export root not found: {exports_root}")

available_dirs = {path.name: path for path in exports_root.iterdir() if path.is_dir()}

include_set = set(INCLUDE_INSTANCES or [])
exclude_set = set(EXCLUDE_INSTANCES or [])

selected_appliances: List[Dict[str, Any]] = []
skipped_missing: List[str] = []
skipped_filtered: List[str] = []

for appliance in available_appliances:
    name = appliance["name"]
    if include_set and name not in include_set:
        skipped_filtered.append(name)
        continue
    if name in exclude_set:
        skipped_filtered.append(name)
        continue
    export_dir = available_dirs.get(name)
    if not export_dir:
        skipped_missing.append(name)
        continue
    selected_appliances.append({
        "name": name,
        "target": appliance.get("target") or name,
        "export_dir": export_dir,
        "token": appliance.get("token"),
        "api_version": appliance.get("api_version") or "v1.14",
        "verify_ssl": appliance.get("verify_ssl", True),
    })

print(f"Repo root     : {REPO_ROOT}")
print(f"Config path   : {CONFIG_PATH}")
print(f"Exports root  : {exports_root}")
print(f"Appliances in config : {[a['name'] for a in available_appliances]}")
print(f"Raw export dirs      : {sorted(available_dirs)}")
print(f"Selected appliances  : {[a['name'] for a in selected_appliances]}")
if skipped_missing:
    print(f"Missing export folders: {skipped_missing}")
if skipped_filtered:
    print(f"Skipped by filter     : {skipped_filtered}")

if not selected_appliances:
    raise RuntimeError("No appliances selected for processing – check raw exports and filters.")


In [ ]:
INVALID_STRINGS = {"", "none", "nan", "null"}
METADATA_COLUMNS = ["Appliance Target", "Appliance Name", "Query Title"]


def is_missing(value) -> bool:
    """
    Treat common null-like values consistently across CSV inputs.
    
    This function is foundational for reliable joins because Discovery exports mix None/NaN/string placeholders.
    """
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    if isinstance(value, str):
        return value.strip().lower() in INVALID_STRINGS
    return False


def to_clean_str(value):
    """
    Convert a value to a trimmed string, returning None for missing values.
    
    Using one normalization path prevents subtle comparison issues in joins and lookups.
    """
    if is_missing(value):
        return None
    return str(value).strip()


def parse_listish(value):
    """
    Parse list-like cell values (real list, serialized list, or scalar) into a clean list.
    
    Discovery exports often store arrays as strings; this normalizes them for aggregation.
    """
    if isinstance(value, list):
        return [str(v).strip() for v in value if not is_missing(v)]
    if is_missing(value):
        return []
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = literal_eval(text)
        except (ValueError, SyntaxError):
            return [text]
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if not is_missing(v)]
    return [text]


def merge_lists(series) -> List[str]:
    """
    Merge a series of list values into a unique, sorted list.
    
    This preserves useful multi-value context (names/IPs/runs) without duplicates.
    """
    merged: List[str] = []
    for values in series:
        if not values:
            continue
        if isinstance(values, list):
            merged.extend(values)
    return sorted({item for item in merged if not is_missing(item)})


def gather_values(row, columns) -> List[str]:
    """
    Collect and deduplicate values from multiple columns in one row.
    
    This is used to consolidate identity evidence spread across several related fields.
    """
    collected: List[str] = []
    for col in columns:
        if col not in row:
            continue
        cell = row[col]
        if isinstance(cell, list):
            collected.extend(cell)
        elif not is_missing(cell):
            collected.append(str(cell).strip())
    return sorted({item for item in collected if not is_missing(item)})


def clean_uuid(value):
    """
    Normalize UUID-like values to their trailing token in lowercase form.
    
    Canonical UUID keys are required for deterministic credential joins.
    """
    text = to_clean_str(value)
    if text is None:
        return None
    return text.split("/")[-1].lower()


def to_bool(value) -> bool:
    """
    Safely coerce mixed boolean representations to True/False.
    
    Discovery data can represent flags as strings, integers, or booleans.
    """
    if isinstance(value, bool):
        return value
    text = to_clean_str(value)
    if text is None:
        return False
    lowered = text.lower()
    if lowered in {"true", "1", "yes"}:
        return True
    if lowered in {"false", "0", "no"}:
        return False
    return False


def collect_unique(series) -> List[str]:
    """
    Return unique, sorted, non-missing scalar values from a series.
    
    Used for stable rollups such as all endpoints, runs, and credentials per identity.
    """
    return sorted({str(v).strip() for v in series if not is_missing(v)})


def drop_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove standard export metadata columns when present.
    
    Keeping only data columns simplifies downstream transformations and matching logic.
    """
    return df.drop(columns=[c for c in METADATA_COLUMNS if c in df.columns], errors="ignore")


def load_csv(path: Path, required: bool = False) -> pd.DataFrame:
    """
    Load a CSV into a DataFrame with optional required-file enforcement.
    
    Optional mode allows graceful degradation; required mode fails fast for core inputs.
    """
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing required CSV file: {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Failed to load required CSV {path}: {exc}")
        print(f"Warning: failed to load optional CSV {path}: {exc}")
        return pd.DataFrame()


def ensure_columns(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """
    Guarantee required columns exist by adding missing ones as None.
    
    This keeps transformation code resilient to schema drift across export versions.
    """
    for col in columns:
        if col not in df.columns:
            df[col] = None
    return df


### Identity Constants and Output Path

Constants used for identity aggregation and the output directory helper.


In [ ]:
IDENTITY_IP_COLUMNS = [
    "DiscoveryAccess.endpoint",
    "Endpoint.endpoint",
    "DiscoveredIPAddress.ip_addr",
    "InferredElement.__all_ip_addrs",
    "NetworkInterface.ip_addr",
]
IDENTITY_NAME_COLUMNS = [
    "InferredElement.name",
    "InferredElement.hostname",
    "InferredElement.local_fqdn",
    "InferredElement.sysname",
    "NetworkInterface.fqdns",
]


def build_output_dir(target: str) -> Path:
    """
    Return the per-appliance output directory path for report artifacts.
    
    The name is sanitized so targets with dots/ports produce filesystem-safe directories.
    """
    sanitized = (target or "unknown").replace(".", "_").replace(":", "_").replace("/", "_")
    if OUTPUT_BASE_DIR is None:
        return REPO_ROOT / f"output_{sanitized}"
    base_root = OUTPUT_BASE_DIR if isinstance(OUTPUT_BASE_DIR, Path) else Path(OUTPUT_BASE_DIR)
    base_root = base_root.expanduser().resolve()
    return base_root / f"output_{sanitized}"




### Identity Aggregation

Builds the identity table from list-like endpoint/name/IP fields.


In [ ]:
def build_identity_table(id_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build one identity row per originating endpoint with aggregated IP/name lists.
    
    This creates the stable identity anchor used to group many DiscoveryAccess records.
    """
    if id_df is None or id_df.empty:
        return pd.DataFrame(columns=["Identities.endpoint", "list_of_ips", "list_of_names"])

    work = id_df.copy()
    length = len(work)
    for col in IDENTITY_IP_COLUMNS + IDENTITY_NAME_COLUMNS:
        if col in work.columns:
            work[col] = work[col].apply(parse_listish)
        else:
            work[col] = [[] for _ in range(length)]

    primary: List[Optional[str]] = []
    endpoint_series = work["Endpoint.endpoint"] if "Endpoint.endpoint" in work.columns else pd.Series([[] for _ in range(length)])
    for access_list, endpoint_list in zip(work["DiscoveryAccess.endpoint"], endpoint_series):
        candidate = None
        for value in (access_list or []) + (endpoint_list or []):
            clean = to_clean_str(value)
            if clean:
                candidate = clean
                break
        primary.append(candidate)
    work["Identities.endpoint"] = primary
    work = work[work["Identities.endpoint"].notna()].copy()

    work["ips_tmp"] = work.apply(lambda row: gather_values(row, IDENTITY_IP_COLUMNS), axis=1)
    work["names_tmp"] = work.apply(lambda row: gather_values(row, IDENTITY_NAME_COLUMNS), axis=1)

    return (
        work.groupby("Identities.endpoint", dropna=False)
        .agg({"ips_tmp": merge_lists, "names_tmp": merge_lists})
        .reset_index()
        .rename(columns={"ips_tmp": "list_of_ips", "names_tmp": "list_of_names"})
    )




### Time and Consistency Helpers

Parses timestamps and computes consistency labels (`Always`, `Usually`, `Most Often`).


In [ ]:
def parse_scan_end_ts(value) -> pd.Timestamp:
    """
    Parse scan end timestamps into timezone-aware UTC pandas timestamps.
    
    A consistent timestamp type is needed for reliable latest-record ranking.
    """
    text = to_clean_str(value)
    if not text:
        return pd.NaT
    return pd.to_datetime(text, utc=True, errors="coerce")


def compute_consistency(states: List[Optional[str]]) -> Optional[str]:
    """
    Classify endpoint end-state consistency as Always/Usually/Most Often.
    
    This provides a human-readable stability signal over historical scan outcomes.
    """
    cleaned = [to_clean_str(s) for s in states if to_clean_str(s)]
    if not cleaned:
        return None
    total = len(cleaned)
    counts: Dict[str, int] = {}
    for state in cleaned:
        counts[state] = counts.get(state, 0) + 1
    top = max(counts, key=counts.get)
    if counts[top] == total:
        return f"Always {top}"
    if counts[top] >= total - 2:
        return f"Usually {top}"
    return f"Most Often {top}"


def normalize_outpost_id(value) -> Optional[str]:
    """
    Normalize outpost identifiers to a canonical ID token.
    
    Canonical IDs allow joining run data and fallback outpost lookup maps safely.
    """
    text = to_clean_str(value)
    if text is None:
        return None
    return text.split("/")[-1]




### Outpost Mapping Helper

Loads optional outpost ID-to-name fallback mappings from CSV.


In [ ]:
def load_outpost_lookup(export_dir: Path) -> Dict[str, str]:
    """
    Load optional outpost ID -> outpost name fallback mappings from CSV.
    
    This enriches run records when direct outpost_name fields are missing.
    """
    outpost_map_csv = export_dir / "discoveryrun_outpost_map.csv"
    df = drop_metadata(load_csv(outpost_map_csv, required=False))
    if df.empty:
        return {}

    id_col = None
    name_col = None
    for candidate in ["outpost_id", "DiscoveryRun.outpost_id", "Outpost ID", "Outpost.id"]:
        if candidate in df.columns:
            id_col = candidate
            break
    for candidate in ["outpost_name", "DiscoveryRun.outpost_name", "Outpost Name", "Outpost.name"]:
        if candidate in df.columns:
            name_col = candidate
            break

    if not id_col or not name_col:
        return {}

    mapping: Dict[str, str] = {}
    for _, row in df.iterrows():
        oid = normalize_outpost_id(row.get(id_col))
        name = to_clean_str(row.get(name_col))
        if oid and name:
            mapping[oid] = name
    return mapping




### Credential CSV Fallback

Loads credential label/username mappings from `credential_success.csv` when available.


In [ ]:
def load_credential_csv_map(output_dir: Path) -> Dict[str, Dict[str, Optional[str]]]:
    """
    Load credential UUID -> label/username mappings from credential_success.csv.
    
    This serves as an offline fallback when live vault API access is unavailable.
    """
    path = output_dir / "credential_success.csv"
    df = drop_metadata(load_csv(path, required=False))
    if df.empty:
        return {}

    uuid_col = None
    for candidate in ["uuid", "UUID", "credential_uuid", "Credential UUID"]:
        if candidate in df.columns:
            uuid_col = candidate
            break
    if not uuid_col:
        return {}

    label_col = "label" if "label" in df.columns else ("Credential" if "Credential" in df.columns else None)
    username_col = "username" if "username" in df.columns else ("Login ID" if "Login ID" in df.columns else None)

    mapping: Dict[str, Dict[str, Optional[str]]] = {}
    for _, row in df.iterrows():
        key = clean_uuid(row.get(uuid_col))
        if not key:
            continue
        mapping[key] = {
            "label": to_clean_str(row.get(label_col)) if label_col else None,
            "username": to_clean_str(row.get(username_col)) if username_col else None,
        }
    return mapping




### Vault Credential Lookup

Queries vault credentials through Tideway and builds a UUID-keyed map.


In [ ]:
def load_vault_credential_map(instance: Dict[str, Any]) -> Dict[str, Dict[str, Optional[str]]]:
    """
    Fetch vault credential metadata via Tideway and build a UUID-keyed lookup.
    
    This is the authoritative source for label/username alignment with credential UUIDs.
    """
    if tideway is None:
        print("Vault lookup skipped: tideway not installed")
        return {}

    token = to_clean_str(instance.get("token"))
    if not token:
        print(f"Vault lookup skipped for {instance.get('name')}: missing token")
        return {}

    target = to_clean_str(instance.get("target"))
    if not target:
        return {}

    api_version = str(instance.get("api_version") or "v1.14").lstrip("vV")
    verify_ssl = bool(instance.get("verify_ssl", True))

    try:
        app = tideway.appliance(target, token, api_version=api_version, ssl_verify=verify_ssl)
        creds_api = app.credentials()
        payload = creds_api.get_vault_credentials.json()
    except Exception as exc:
        print(f"Vault lookup failed for {target}: {exc}")
        return {}

    mapping: Dict[str, Dict[str, Optional[str]]] = {}
    for cred in payload or []:
        if not isinstance(cred, dict):
            continue
        key = clean_uuid(cred.get("uuid"))
        if not key:
            continue
        username = (
            cred.get("username")
            or cred.get("snmp.v3.securityname")
            or cred.get("aws.access_key_id")
            or cred.get("azure.application_id")
        )
        mapping[key] = {
            "label": to_clean_str(cred.get("label")),
            "username": to_clean_str(username),
        }

    return mapping




### Credential Lookup Merge

Combines CSV fallback and vault mappings into one credential lookup map.


In [ ]:
def build_credential_lookup(instance: Dict[str, Any], output_dir: Path) -> Dict[str, Dict[str, Optional[str]]]:
    """
    Merge CSV fallback and vault mappings into one credential lookup dictionary.
    
    Vault entries override CSV data so the freshest credential metadata is preferred.
    """
    csv_map = load_credential_csv_map(output_dir)
    vault_map = load_vault_credential_map(instance)

    merged = dict(csv_map)
    merged.update(vault_map)
    print(f"Credential lookup entries: csv={len(csv_map)} vault={len(vault_map)} merged={len(merged)}")
    return merged




### Instance Processing

Main per-instance processing pipeline and CSV export logic.


In [ ]:
def process_instance(instance: Dict[str, Any]) -> Dict[str, Any]:
    """
    Run the full device-analysis pipeline for one appliance export directory.
    
    This function loads inputs, performs joins/aggregations, enriches credentials/outposts, and writes device_analysis.csv.
    """
    name = instance["name"]
    target = instance.get("target") or name
    export_dir: Path = instance["export_dir"]
    print(f"=== Processing {name} ({target}) ===")

    identities_csv = export_dir / "devices_report_identities.csv"
    last_discovery_csv = export_dir / "devices_report_last_discovery.csv"
    key_csv = export_dir / "discovery_analysis_key_map.csv"
    access_csv = export_dir / "discovery_analysis_access_summary.csv"
    device_csv = export_dir / "discovery_analysis_deviceinfo.csv"
    run_csv = export_dir / "discovery_analysis_discovery_runs.csv"
    dropped_csv = export_dir / "discovery_analysis_dropped_endpoints.csv"

    required = [identities_csv, last_discovery_csv, key_csv, access_csv, device_csv, run_csv]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        print(f"Missing required CSV files for {name}: {missing}")
        return {"instance": name, "target": target, "output_path": None, "rows": 0, "status": "missing_csv"}

    output_dir = build_output_dir(target)
    output_dir.mkdir(parents=True, exist_ok=True)

    credential_lookup = build_credential_lookup(instance, output_dir)
    outpost_lookup = load_outpost_lookup(export_dir)

    id_df = drop_metadata(load_csv(identities_csv, required=True))
    ld_df = drop_metadata(load_csv(last_discovery_csv, required=True))
    key_df = drop_metadata(load_csv(key_csv, required=True))
    access_df = drop_metadata(load_csv(access_csv, required=True))
    device_df = drop_metadata(load_csv(device_csv, required=True))
    run_df = drop_metadata(load_csv(run_csv, required=True))
    dropped_df = drop_metadata(load_csv(dropped_csv, required=False))

    print(f"Identity rows: {len(id_df)}")
    print(f"Last discovery rows: {len(ld_df)}")
    print(f"Key rows: {len(key_df)} | Access rows: {len(access_df)} | Device rows: {len(device_df)} | Run rows: {len(run_df)}")

    identities_df = build_identity_table(id_df)
    print(f"Unique identity endpoints: {len(identities_df)}")

    ensure_columns(ld_df, [
        "DiscoveryAccess.endpoint",
        "DiscoveryAccess.scan_endtime_raw",
        "DiscoveryRun.label",
        "DiscoveryAccess.host_node_updated",
    ])
    ld_df["DiscoveryAccess.host_node_updated"] = ld_df["DiscoveryAccess.host_node_updated"].apply(to_bool)

    ld_map3: Dict[Tuple[str, str, str], bool] = {}
    ld_map2: Dict[Tuple[str, str], bool] = {}
    for _, row in ld_df.iterrows():
        ep = to_clean_str(row.get("DiscoveryAccess.endpoint"))
        scan_raw = to_clean_str(row.get("DiscoveryAccess.scan_endtime_raw"))
        run_label = to_clean_str(row.get("DiscoveryRun.label"))
        success = to_bool(row.get("DiscoveryAccess.host_node_updated"))
        if ep and scan_raw:
            ld_map2[(ep, scan_raw)] = ld_map2.get((ep, scan_raw), False) or success
            if run_label:
                ld_map3[(ep, scan_raw, run_label)] = ld_map3.get((ep, scan_raw, run_label), False) or success

    key_df = ensure_columns(key_df, [
        "DiscoveryAccess.id",
        "DiscoveryAccess.previous_id",
        "DiscoveryAccess.next_id",
        "DeviceInfo.id",
        "DiscoveryRun.id",
    ])

    access_df = ensure_columns(access_df, [
        "DiscoveryAccess.id",
        "DiscoveryAccess.endpoint",
        "DiscoveryAccess.scan_starttime",
        "DiscoveryAccess.scan_endtime",
        "DiscoveryAccess.scan_endtime_raw",
        "DiscoveryAccess.end_state",
        "DiscoveryAccess.reason_not_updated",
        "DiscoveryAccess.result",
        "DiscoveryAccess.node_kind",
        "DiscoveryAccess.host_node_updated",
    ])

    device_df = ensure_columns(device_df, [
        "DeviceInfo.id",
        "DeviceInfo.hostname",
        "DeviceInfo.kind",
        "DeviceInfo.inferred_kind",
        "DeviceInfo.last_access_method",
        "DeviceInfo.last_credential",
        "DeviceInfo.last_slave",
        "DeviceInfo.probed_os",
    ])

    run_df = ensure_columns(run_df, [
        "DiscoveryRun.id",
        "DiscoveryRun.label",
        "DiscoveryRun.starttime",
        "DiscoveryRun.endtime",
        "DiscoveryRun.outpost_id",
        "DiscoveryRun.outpost_name",
    ])

    key_join = key_df[["DiscoveryAccess.id", "DeviceInfo.id", "DiscoveryRun.id"]].copy()
    key_join = key_join.dropna(subset=["DiscoveryAccess.id"]).drop_duplicates()
    if key_join.duplicated(subset=["DiscoveryAccess.id"]).any():
        key_join = key_join.drop_duplicates(subset=["DiscoveryAccess.id"], keep="first")

    merged_access = access_df.copy()
    merged_access = merged_access.merge(key_join, on="DiscoveryAccess.id", how="left")
    merged_access = merged_access.merge(device_df, on="DeviceInfo.id", how="left")
    merged_access = merged_access.merge(run_df, on="DiscoveryRun.id", how="left")

    if merged_access.empty:
        print("No merged DiscoveryAccess rows available; writing empty output")
        empty_cols = [
            "last_scanned_ip",
            "last_identity",
            "last_kind",
            "all_device_names",
            "all_endpoints",
            "all_credentials_used",
            "all_discovery_runs",
            "last_credential",
            "last_credential_label",
            "last_credential_username",
            "last_start_time",
            "last_run",
            "last_endstate",
            "last_result",
            "last_access_method",
            "current_access",
            "reason_not_updated",
            "consistency",
            "outpost_id",
            "outpost_name",
            "last_successful_identity",
            "last_successful_ip",
            "last_successful_credential",
            "last_successful_credential_label",
            "last_successful_credential_username",
            "last_successful_start_time",
            "last_successful_run",
            "last_successful_endstate",
        ]
        df_out = pd.DataFrame(columns=empty_cols)
        df_out.insert(0, "Discovery Instance", target)
        output_csv = output_dir / OUTPUT_FILENAME
        df_out.to_csv(output_csv, index=False)
        return {"instance": name, "target": target, "output_path": output_csv, "rows": 0, "status": "no-data"}

    if "DiscoveryAccess.node_kind" not in merged_access.columns:
        merged_access["DiscoveryAccess.node_kind"] = None
    merged_access["DiscoveryAccess.node_kind"] = merged_access["DiscoveryAccess.node_kind"].fillna(merged_access.get("DeviceInfo.kind"))
    merged_access["DiscoveryAccess.node_kind"] = merged_access["DiscoveryAccess.node_kind"].fillna(merged_access.get("DeviceInfo.inferred_kind"))

    lam = merged_access.get("DeviceInfo.last_access_method")
    if lam is None:
        lam = pd.Series([None] * len(merged_access), index=merged_access.index, dtype="object")
    lam = lam.astype("object")

    slave = merged_access.get("DeviceInfo.last_slave")
    if slave is None:
        slave = pd.Series([False] * len(merged_access), index=merged_access.index)
    slave = slave.apply(to_bool)

    probed = merged_access.get("DeviceInfo.probed_os")
    if probed is None:
        probed = pd.Series([False] * len(merged_access), index=merged_access.index)
    probed = probed.apply(to_bool)

    cond1 = lam.isin(["windows", "rcmd"]) & slave
    merged_access["DiscoveryAccess.current_access"] = np.where(cond1, lam, np.where(probed, "Probe", lam))
    merged_access["last_access_method"] = merged_access["DiscoveryAccess.current_access"]

    merged_access["last_cred_short"] = merged_access.get("DeviceInfo.last_credential", pd.Series([None] * len(merged_access), index=merged_access.index)).apply(clean_uuid)

    if DEVICES_WITH_CRED_UUID:
        want_uuid = clean_uuid(DEVICES_WITH_CRED_UUID)
        merged_access = merged_access[merged_access["last_cred_short"] == want_uuid].copy()
        print(f"Filtered merged rows to credential {want_uuid} -> {len(merged_access)} rows")

    merged_access["scan_end_ts"] = merged_access.get("DiscoveryAccess.scan_endtime_raw", pd.Series([None] * len(merged_access), index=merged_access.index)).apply(parse_scan_end_ts)
    merged_access["scan_end_rank"] = merged_access["scan_end_ts"].apply(lambda ts: ts.value if pd.notna(ts) else -1)

    base_success = merged_access.get("DiscoveryAccess.host_node_updated")
    if base_success is None:
        base_success = pd.Series([False] * len(merged_access), index=merged_access.index)
    base_success = base_success.apply(to_bool)

    resolved_success: List[bool] = []
    for idx, row in merged_access.iterrows():
        ep = to_clean_str(row.get("DiscoveryAccess.endpoint"))
        scan_raw = to_clean_str(row.get("DiscoveryAccess.scan_endtime_raw"))
        run_label = to_clean_str(row.get("DiscoveryRun.label"))
        current = base_success.loc[idx]
        if ep and scan_raw and run_label and (ep, scan_raw, run_label) in ld_map3:
            current = ld_map3[(ep, scan_raw, run_label)]
        elif ep and scan_raw and (ep, scan_raw) in ld_map2:
            current = ld_map2[(ep, scan_raw)]
        resolved_success.append(to_bool(current))
    merged_access["DiscoveryAccess.host_node_updated"] = pd.Series(resolved_success, index=merged_access.index)

    if "DiscoveryRun.outpost_id" not in merged_access.columns:
        merged_access["DiscoveryRun.outpost_id"] = None
    if "DiscoveryRun.outpost_name" not in merged_access.columns:
        merged_access["DiscoveryRun.outpost_name"] = None

    def fill_outpost_name(row):
        """
        Fill missing run outpost names using the optional ID->name fallback map.
        
        This preserves outpost context even when upstream run exports omit names.
        """
        existing = to_clean_str(row.get("DiscoveryRun.outpost_name"))
        if existing:
            return existing
        outpost_id = normalize_outpost_id(row.get("DiscoveryRun.outpost_id"))
        if outpost_id and outpost_id in outpost_lookup:
            return outpost_lookup[outpost_id]
        return None

    merged_access["DiscoveryRun.outpost_name"] = merged_access.apply(fill_outpost_name, axis=1)
    merged_access["DiscoveryRun.outpost_id"] = merged_access["DiscoveryRun.outpost_id"].apply(normalize_outpost_id)

    endpoint_states: Dict[str, List[Optional[str]]] = {}
    for _, row in merged_access.iterrows():
        ep = to_clean_str(row.get("DiscoveryAccess.endpoint"))
        if not ep:
            continue
        endpoint_states.setdefault(ep, []).append(row.get("DiscoveryAccess.end_state"))

    if not dropped_df.empty:
        dropped_df = ensure_columns(dropped_df, ["Endpoint", "End_State"])
        for _, row in dropped_df.iterrows():
            ep = to_clean_str(row.get("Endpoint"))
            if not ep:
                continue
            endpoint_states.setdefault(ep, []).append(row.get("End_State"))

    consistency_map = {ep: compute_consistency(states) for ep, states in endpoint_states.items()}

    ip_map = identities_df[["Identities.endpoint", "list_of_ips"]].explode("list_of_ips", ignore_index=True)
    ip_map["list_of_ips"] = ip_map["list_of_ips"].apply(to_clean_str)
    ip_map = ip_map.dropna(subset=["list_of_ips"])

    merged = ip_map.merge(
        merged_access,
        left_on="list_of_ips",
        right_on="DiscoveryAccess.endpoint",
        how="left",
    )

    if "last_cred_short" not in merged.columns:
        merged["last_cred_short"] = None
    merged["last_cred_short"] = merged["last_cred_short"].apply(clean_uuid)

    if "scan_end_rank" not in merged.columns:
        merged["scan_end_rank"] = -1
    else:
        merged["scan_end_rank"] = merged["scan_end_rank"].fillna(-1)

    print(f"Merged rows: {len(merged)}")

    if merged.empty:
        df_out = pd.DataFrame(columns=[
            "last_scanned_ip",
            "last_identity",
            "last_kind",
            "all_device_names",
            "all_endpoints",
            "all_credentials_used",
            "all_discovery_runs",
            "last_credential",
            "last_credential_label",
            "last_credential_username",
            "last_start_time",
            "last_run",
            "last_endstate",
            "last_result",
            "last_access_method",
            "current_access",
            "reason_not_updated",
            "consistency",
            "outpost_id",
            "outpost_name",
            "last_successful_identity",
            "last_successful_ip",
            "last_successful_credential",
            "last_successful_credential_label",
            "last_successful_credential_username",
            "last_successful_start_time",
            "last_successful_run",
            "last_successful_endstate",
        ])
        df_out.insert(0, "Discovery Instance", target)
        output_csv = output_dir / OUTPUT_FILENAME
        df_out.to_csv(output_csv, index=False)
        return {"instance": name, "target": target, "output_path": output_csv, "rows": 0, "status": "no-data"}

    grp = merged.groupby("Identities.endpoint", dropna=False)

    agg_device_names = grp["DeviceInfo.hostname"].apply(collect_unique).reset_index(name="all_device_names")
    agg_endpoints = grp["DiscoveryAccess.endpoint"].apply(collect_unique).reset_index(name="all_endpoints")
    agg_runs = grp["DiscoveryRun.label"].apply(collect_unique).reset_index(name="all_discovery_runs")
    agg_creds = grp["last_cred_short"].apply(collect_unique).reset_index(name="all_credentials_used")

    latest_idx = grp["scan_end_rank"].idxmax()
    latest = merged.loc[latest_idx.tolist(), [
        "Identities.endpoint",
        "DiscoveryAccess.endpoint",
        "DeviceInfo.hostname",
        "DiscoveryAccess.node_kind",
        "last_cred_short",
        "DiscoveryAccess.scan_starttime",
        "DiscoveryRun.label",
        "DiscoveryAccess.end_state",
        "DiscoveryAccess.result",
        "DiscoveryAccess.current_access",
        "last_access_method",
        "DiscoveryAccess.reason_not_updated",
        "DiscoveryRun.outpost_id",
        "DiscoveryRun.outpost_name",
    ]].copy()

    latest = latest.rename(columns={
        "DiscoveryAccess.endpoint": "last_scanned_ip",
        "DeviceInfo.hostname": "last_identity",
        "DiscoveryAccess.node_kind": "last_kind",
        "last_cred_short": "last_credential",
        "DiscoveryAccess.scan_starttime": "last_start_time",
        "DiscoveryRun.label": "last_run",
        "DiscoveryAccess.end_state": "last_endstate",
        "DiscoveryAccess.result": "last_result",
        "DiscoveryAccess.current_access": "current_access",
        "DiscoveryAccess.reason_not_updated": "reason_not_updated",
        "DiscoveryRun.outpost_id": "outpost_id",
        "DiscoveryRun.outpost_name": "outpost_name",
    })

    latest["last_access_method"] = latest["current_access"]
    latest["consistency"] = latest["last_scanned_ip"].map(consistency_map)

    def credential_field(uuid_value, field_name: str):
        """
        Resolve a credential attribute (label/username) by normalized UUID key.
        
        Using one accessor ensures latest and last-successful credential fields stay aligned.
        """
        key = clean_uuid(uuid_value)
        if not key:
            return None
        return to_clean_str((credential_lookup.get(key) or {}).get(field_name))

    latest["last_credential_label"] = latest["last_credential"].map(lambda v: credential_field(v, "label"))
    latest["last_credential_username"] = latest["last_credential"].map(lambda v: credential_field(v, "username"))

    succ = merged[merged.get("DiscoveryAccess.host_node_updated") == True].copy()
    if succ.empty:
        last_succ = pd.DataFrame(columns=[
            "Identities.endpoint",
            "last_successful_identity",
            "last_successful_ip",
            "last_successful_credential",
            "last_successful_start_time",
            "last_successful_run",
            "last_successful_endstate",
            "last_successful_credential_label",
            "last_successful_credential_username",
        ])
    else:
        succ_idx = succ.groupby("Identities.endpoint", dropna=False)["scan_end_rank"].idxmax()
        last_succ = succ.loc[succ_idx.tolist(), [
            "Identities.endpoint",
            "DeviceInfo.hostname",
            "DiscoveryAccess.endpoint",
            "last_cred_short",
            "DiscoveryAccess.scan_starttime",
            "DiscoveryRun.label",
            "DiscoveryAccess.end_state",
        ]].copy()
        last_succ = last_succ.rename(columns={
            "DeviceInfo.hostname": "last_successful_identity",
            "DiscoveryAccess.endpoint": "last_successful_ip",
            "last_cred_short": "last_successful_credential",
            "DiscoveryAccess.scan_starttime": "last_successful_start_time",
            "DiscoveryRun.label": "last_successful_run",
            "DiscoveryAccess.end_state": "last_successful_endstate",
        })
        last_succ["last_successful_credential_label"] = last_succ["last_successful_credential"].map(lambda v: credential_field(v, "label"))
        last_succ["last_successful_credential_username"] = last_succ["last_successful_credential"].map(lambda v: credential_field(v, "username"))

    df_out = (
        latest
        .merge(agg_device_names, on="Identities.endpoint", how="left")
        .merge(agg_endpoints, on="Identities.endpoint", how="left")
        .merge(agg_runs, on="Identities.endpoint", how="left")
        .merge(agg_creds, on="Identities.endpoint", how="left")
        .merge(last_succ, on="Identities.endpoint", how="left")
        .merge(identities_df[["Identities.endpoint", "list_of_names"]], on="Identities.endpoint", how="left")
    )

    def fallback_identity(row):
        """
        Backfill last_identity from aggregated identity names when hostname is missing.
        
        This reduces empty identity labels for rows where scan-level hostname is unavailable.
        """
        primary = to_clean_str(row.get("last_identity"))
        if primary:
            return primary
        names = row.get("list_of_names")
        if isinstance(names, list) and names:
            return names[0]
        return None

    df_out["last_identity"] = df_out.apply(fallback_identity, axis=1)

    desired_columns = [
        "last_scanned_ip",
        "last_identity",
        "last_kind",
        "all_device_names",
        "all_endpoints",
        "all_credentials_used",
        "all_discovery_runs",
        "last_credential",
        "last_credential_label",
        "last_credential_username",
        "last_start_time",
        "last_run",
        "last_endstate",
        "last_result",
        "last_access_method",
        "current_access",
        "reason_not_updated",
        "consistency",
        "outpost_id",
        "outpost_name",
        "last_successful_identity",
        "last_successful_ip",
        "last_successful_credential",
        "last_successful_credential_label",
        "last_successful_credential_username",
        "last_successful_start_time",
        "last_successful_run",
        "last_successful_endstate",
    ]

    for col in desired_columns:
        if col not in df_out.columns:
            df_out[col] = None

    df_out = df_out[desired_columns]
    df_out.insert(0, "Discovery Instance", target)

    output_csv = output_dir / OUTPUT_FILENAME
    df_out.to_csv(output_csv, index=False)

    print(f"Output rows: {len(df_out)} | Saved to {output_csv}")
    display(df_out.head(5))

    return {
        "instance": name,
        "target": target,
        "output_path": output_csv,
        "rows": int(len(df_out)),
        "status": "ok",
    }


In [ ]:
results = []
for appliance in selected_appliances:
    outcome = process_instance(appliance)
    results.append(outcome)

summary_df = pd.DataFrame(results)
display(summary_df)
